In [ ]:
import logging
from pathlib import Path
from dotenv import dotenv_values
import logging

In [ ]:
from damo_afvoergebiedaanvoergebied import \
    run_generator_duikers, \
    run_generator_gebiedsorde, \
    run_generator_afvoergebieden, \
    run_generator_specifieke_afvoer

In [ ]:
%load_ext autoreload
%autoreload 2

Voorbeeld: Op deze wijze zouden de workflows aangeroepen kunnen worden. Echter is het handiger om een bepaalde volgorde aan te houden met verschillende instellingen, omdat er een bepaalde iteratie nodig is. De gebiedsorde wordt gebruikt om te bepalen welke watergangen zijn aangesloten, de overige watergangen (zaksloten) hoeven niet meegenomen worden in de afvoergebieden.

In [ ]:
# toml_path = "settings_vallei_en_veluwe.toml"
# run_generator_duikers(path=toml_path)
# run_generator_gebiedsorde(path=toml_path)
# run_generator_afvoergebieden(path=toml_path)
# run_generator_specifieke_afvoer(path=toml_path)

Stap 0: Opgeven locatie TOML-bestand met instellingen. De folders opgegeven voor basisdata en resultaten in het TOML-bestand zijn relatief ten opzichte van de locatie waar dit bestand staat. Dus je hebt bijvoorbeeld een mappenstructuur als volgt:
- 0_basisdata\
- 1_resultaten\
- settings_simulatie.toml

In [ ]:
# load base directory from .env file (for collaboration purposes)
config = dotenv_values("..\\.env")
base_dir = Path(config["PATH_BASIS"])
# path_toml_settings = Path(base_dir, "leuvenumse_beek", "settings_leuvenumse_beek.toml")
path_toml_settings = Path(base_dir, "vallei_en_veluwe", "settings_vallei_en_veluwe.toml")

Stap 1: We starten met het verbinden van de C-watergangen (overige watergangen) met de hydroobjecten (A/B-watergangen).

In [ ]:
generator = run_generator_duikers(
    path=path_toml_settings
)

Stap 2: Workflow gebiedsorde (voor de eerste keer) om te bepalen welke hoofdwatergangen meedoen. 
- generate_order_no: order nummer bepalen (2,3,4,5,...)
- generate_order_code: order code voor hoofdwatergangen (hydroobjecten of A/B)
- generate_order_code_overig: order code voor overige watergangen (overige_watergangen of C)

In [ ]:
order = run_generator_gebiedsorde(
    path=path_toml_settings,
    generate_order_no=True,
    generate_order_code=True,
    generate_order_code_overig=True,
)

Stap 3: Workflow afvoergebieden. Voor alle watergangen en RWS-wateren worden er dan afvoergebieden gegenereerd. Omdat we nog niet de complete postprocessing doen (het aggregeren op basis van orde-codering, doen we niet alle stappen):
- preprocess: bool      - omzetten GHG naar juiste resolutie, inbranden watergangen en rws-wateren (alleen uniek nummer)
- process: bool         - genereren van de afwateringseenheden per watergang met nummering van de watergangen/rws-wateren
- postprocess1: bool    - koppelen van de afwateringseenheden aan de hoofdwatergangen en de orde-code (incl aggregatie naar dat niveau)
- postprocess2: bool    - aggregeren naar (deel)stroomgebieden

Omdat we de orde-codering nog gaan aanpassen op basis van de hierna afgeleide afvoer, doen we de post-processing nog niet.

In [ ]:
afvoergebieden = run_generator_afvoergebieden(
    path=path_toml_settings,
    preprocess=True,
    process=True,
    postprocess1=False,
    postprocess2=False,
)

Stap 5: Workflow specifieke/maatgevende afvoer: Bereken de maatgevende afvoer voor alle watergangen in het systeem.
- generate_values: sommeren van de waardes uit het opgegeven raster per afwateringseenheid en dat koppelen aan de watergang
- sum_values: afvoer sommeren in benedenstroomse richting waarin splitsingen worden meegenomen.

In [ ]:
afvoer = run_generator_specifieke_afvoer(
    path=path_toml_settings,
    generate_values=True,
    sum_values=True,
)

Stap 6: Workflow gebiedsorde nogmaals: Omdat de maatgevende afvoeren nu bekend zijn, wordt dit meegenomen bij het onderscheid van hoofdwatergangen en zijwatergangen.

In [ ]:
order = run_generator_gebiedsorde(
    path=path_toml_settings,
    generate_order_no=True,
    generate_order_code=True,
    generate_order_code_overig=True,
)

Stap 7: Postprocessing van de afvoergebieden. Aggregeren naar verschillende niveuas en tot en met 2e orde stroomgebieden 

In [ ]:
afvoergebieden = run_generator_afvoergebieden(
    path=path_toml_settings,
    preprocess=False,
    process=False,
    postprocess1=True,
    postprocess2=True,
)